<a href="https://colab.research.google.com/github/iav2002/AppliedDeepLearning/blob/main/Part2_6_InvestigationClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification Investigations

Three classification only investigations will be done here, output layer activation, on or off including global average pooling, and number of fully connected layers. Working hypothesis carried from notebook 5, classification is bottlenecked by overfitting not capacity, so GAP especially should help.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Colab Notebooks/AppliedDL/face_age.zip" /content/
!cp -r "/content/drive/MyDrive/Colab Notebooks/AppliedDL/data_splits" /content/
!unzip -q /content/face_age.zip -d /content/

## 2. Imports

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## 3. Dataset class

Same `FaceAgeDataset` carried over only for classification.

In [8]:
CATEGORIES = ["infant", "child", "teen", "youth", "mid", "mature", "senior"]
CAT_TO_IDX = {c: i for i, c in enumerate(CATEGORIES)}


class FaceAgeDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"/content/{row['path']}").convert("RGB")
        if self.transform:
            img = self.transform(img)
        lbl = torch.tensor(CAT_TO_IDX[row["category"]], dtype=torch.long)
        return img, lbl

## 4. Transforms and dataloaders

Same pipeline, classification only.

In [9]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_ds = FaceAgeDataset("/content/data_splits/train.csv", transform=transform)
val_ds = FaceAgeDataset("/content/data_splits/val.csv", transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

print(len(train_ds), len(val_ds))

7320 1464


## 5. Train and eval functions

In [10]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0
    n_samples = 0

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = loss_fn(preds, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        n_samples += imgs.size(0)
    return total_loss / n_samples


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    n_samples = 0
    correct = 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = model(imgs)
            loss = loss_fn(preds, lbls)
            total_loss += loss.item() * imgs.size(0)
            n_samples += imgs.size(0)
            correct += (preds.argmax(1) == lbls).sum().item()
    return total_loss / n_samples, correct / n_samples

## 6. Variant runner

Same wrapper, max 12 epochs, patience 3.

In [11]:
def run_variant(model, train_loader, val_loader, loss_fn,
                max_epochs=12, patience=3, lr=1e-3, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_acc = -float("inf")
    best_state = None
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    no_improve = 0

    for epoch in range(max_epochs):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if verbose:
            print(f"epoch {epoch+1:2d}  train {train_loss:.4f}  val {val_loss:.4f}  acc {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose:
                    print(f"early stop at epoch {epoch+1}")
                break

    return {"best_acc": best_acc, "best_state": best_state, "history": history}